# Gemma 4 E4B — Text-to-SQL Fine-Tuning on BIRD

**Run this notebook on Google Colab with T4 GPU**

Runtime → Change runtime type → T4 GPU

This notebook runs all 6 steps in sequence.

**Paths:** On your machine, if the BIRD train bundle lives under `Data/train/train/` (with `train.json` and `train_databases/`), the project uses it automatically. On Colab, Step 1 downloads into `data/bird/` instead.

In [2]:
# ── Cell 1: Clone your repo and install dependencies ─────────────────────────
# Replace with your GitHub repo URL after you push the project

!git clone https://github.com/KARAN6550/Finetuned_Gemma4_t2sql.git
%cd Finetuned_Gemma4_t2sql
!pip install -q -r requirements.txt

fatal: destination path 'Finetuned_Gemma4_t2sql' already exists and is not an empty directory.
/content/Finetuned_Gemma4_t2sql
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 122.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ── Cell 2: Mount Google Drive (to save checkpoints — free Colab disconnects) ─
from google.colab import drive
drive.mount('/content/drive')

import os
# Update checkpoint dir to save to Drive
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/gemma4-text2sql/checkpoints'

In [ ]:
# ── Cell 3: Verify GPU ───────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 4: STEP 1 — Download BIRD dataset ───────────────────────────────────
!python scripts/01_download_bird.py

In [ ]:
# ── Cell 5: STEP 2 — Extract schemas from .sqlite files ──────────────────────
!python scripts/02_extract_schemas.py

In [ ]:
# ── Cell 6: STEP 3 — Format dataset into training prompts ────────────────────
!python scripts/03_prepare_dataset.py

In [ ]:
# ── Cell 7: Login to W&B (run once) ──────────────────────────────────────────
# Get your API key from https://wandb.ai/authorize
import wandb
wandb.login()

In [ ]:
# ── Cell 8: STEP 4 — Train (4–6 hours) ───────────────────────────────────────
# Watch your W&B dashboard at https://wandb.ai while this runs
!python scripts/04_train.py

In [ ]:
# ── Cell 9: STEP 5 — Quick eval sanity check (100 examples) ──────────────────
!python scripts/05_evaluate.py --subset 100

In [ ]:
# ── Cell 10: STEP 5 — Full evaluation (1,534 examples, ~1 hour) ──────────────
!python scripts/05_evaluate.py

In [ ]:
# ── Cell 11: STEP 6 — Push to HuggingFace ────────────────────────────────────
# Set your HF token first:
import os
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'   # get from huggingface.co/settings/tokens
!python scripts/06_push_to_hub.py